# **1일차 과제 — API 활용한 AI 호출 및 프롬프트 구성**

* 진료 메모/증상 텍스트를 활용해서, 역할·조건·출력형식을 갖춘 프롬프트를 직접 설계하고 실행해봅니다.

> **제출자: 조태희** · OZ코딩스쿨 AI 헬스케어 부트캠프 6기 · 자연어 처리(NLP) 1일차 과제

**📌 실행 안내**
- 실행 번호와 출력이 있는 셀은 API 키 없이도 돌아가는 셀(라이브러리 불러오기, 데이터 준비, 프롬프트 만들기)이라 미리 실행해 둔 결과입니다.
- 코드 첫 줄에 **🔑 [Colab에서 실행]** 표시가 있는 셀은 OpenAI API 키가 필요해서 아직 실행하지 않았습니다. Colab에서 `api_key.txt`를 업로드하거나(또는 왼쪽 🔑 보안 비밀에 `OPENAI_API_KEY` 등록) **런타임 → 모두 실행**으로 위에서부터 차례대로 실행해 주세요.
- GPT 답변은 실행할 때마다 표현이 조금씩 달라질 수 있습니다.

# **1. 환경준비**

## **(1) 라이브러리**

In [ ]:
# 🔑 [Colab에서 실행] 필요한 라이브러리 설치 (-q: 설치 로그를 짧게 표시)
!pip install langchain-openai langchain_community -q

In [1]:
import os    # 환경변수(API 키 등)를 다루는 기본 모듈
import ast   # 문자열로 된 파이썬 자료형(딕셔너리 등)을 진짜 자료형으로 바꿔주는 모듈

from langchain_openai import ChatOpenAI                          # OpenAI의 GPT 채팅 모델을 LangChain에서 쓰는 클래스
from langchain_core.messages import HumanMessage, SystemMessage  # 메시지 종류: 사용자 요청(Human) / 역할·규칙 지침(System)

## **(2) OpenAI API Key 확인**

In [ ]:
# 🔑 [Colab에서 실행] api_key.txt 파일 업로드
#   - api_key.txt 안에는 OPENAI_API_KEY=sk-... 형식으로 적혀 있어야 합니다.
#   - Colab 왼쪽 🔑(보안 비밀)에 OPENAI_API_KEY를 등록해 두었다면 이 셀은 건너뛰어도 됩니다.
from google.colab import files   # Colab에서 내 컴퓨터 파일을 올리는 기능
uploaded = files.upload()        # 업로드 창이 뜨면 api_key.txt를 선택합니다

In [ ]:
# 🔑 [Colab에서 실행] API 키를 환경변수로 등록합니다 (코드에 키를 직접 적지 않기 위해서예요)
def load_api_keys(filepath="api_key.txt"):
    with open(filepath, "r") as f:                      # 키 파일을 읽기 모드로 엽니다
        for line in f:                                  # 한 줄씩 읽으면서
            line = line.strip()                         # 앞뒤 공백·줄바꿈을 지우고
            if line and "=" in line:                    # 빈 줄이 아니고 "이름=값" 형태라면
                key, value = line.split("=", 1)         # 첫 번째 '='를 기준으로 이름과 값을 나눠서
                os.environ[key.strip()] = value.strip() # 환경변수로 등록합니다

def ensure_api_key(name):
    """환경변수에 키가 없으면 Colab 보안 비밀 → 직접 입력 순서로 채웁니다."""
    if os.environ.get(name):                            # 이미 등록돼 있으면 그대로 사용
        return                                          # 더 할 일이 없으니 함수 종료
    try:
        from google.colab import userdata               # Colab 🔑 보안 비밀 저장소
        os.environ[name] = userdata.get(name)           # 보안 비밀에 등록된 키를 꺼내 등록
    except Exception:                                   # 보안 비밀에 없거나 Colab이 아니면
        from getpass import getpass                     # 입력한 글자가 화면에 안 보이는 입력창
        os.environ[name] = getpass(f"{name}를 입력하세요: ")   # 직접 입력받아 등록

# API 키 로드 및 환경변수 설정
if os.path.exists("api_key.txt"):                       # 업로드한 키 파일이 있으면
    load_api_keys("api_key.txt")                        # 파일에서 읽어서 등록
ensure_api_key("OPENAI_API_KEY")                        # 그래도 없으면 보안 비밀/직접 입력으로 채움

* ⚠️ api_key가 잘 등록되었는지 확인 후, 아래 출력 셀은 실행 후 삭제하세요.

In [ ]:
# 🔑 [Colab에서 실행] 키가 잘 들어왔는지 앞부분만 확인합니다
#   (전체 키가 노트북에 남으면 유출될 수 있어서 앞 7글자만 보이게 했습니다. 확인 후 이 셀은 삭제하세요.)
print(os.environ['OPENAI_API_KEY'][:7] + "...")

In [ ]:
# 🔑 [Colab에서 실행] GPT 모델 준비 (API 키가 있어야 객체를 만들 수 있습니다)
chat = ChatOpenAI(model="gpt-4.1-mini", temperature=0)   # temperature=0 → 무작위성을 최소로 해서 매번 비슷한 답이 나오게 함

# **2. 과제 데이터**

* 아래 진료 메모 2건과 증상 텍스트 1건을 문제에서 공통으로 사용합니다.

In [2]:
# 진료 메모 1: 고혈압 환자 (""" """ 세 개 따옴표는 여러 줄 문자열을 만들 때 씁니다)
memo_1 = """
[진료 메모 - 고혈압]
환자는 3년 전 고혈압 진단을 받았으며, 현재 하루 1회 혈압약을 복용 중이다.
최근 방문 시 측정한 혈압은 148/92mmHg로 목표 수치보다 다소 높게 나타났다.
환자는 최근 업무 스트레스가 많고, 야근이 잦아 규칙적인 식사와 운동이 어렵다고 호소했다.
염분 섭취를 줄이고 주 3회 이상 유산소 운동을 병행할 것을 권고하였으며, 2주 후 재방문하여
혈압 변화를 다시 확인하기로 하였다.
"""

# 진료 메모 2: 당뇨 환자
memo_2 = """
[진료 메모 - 당뇨]
제2형 당뇨병으로 5년째 경구약을 복용 중인 환자로, 공복혈당은 138mg/dL로 확인되었다.
최근 당화혈색소(HbA1c)는 7.4%로 지난 방문 대비 소폭 상승하였다.
환자는 최근 야식과 단 음료 섭취가 늘었다고 답했으며, 혈당 자가측정은 주 1~2회에 그치고 있다.
탄수화물 섭취량 조절과 혈당 자가측정 빈도를 늘릴 것을 권고하였다.
"""

# 증상 텍스트 (줄 끝의 \ 는 "다음 줄과 이어서 한 줄로 본다"는 뜻)
symptom_text = "3일 전부터 기침과 가래가 심해졌고, 어제부터는 미열(37.8도)과 함께 목이 붓고\
 아픈 느낌이 있습니다. 콧물은 없고, 숨쉬기 답답한 느낌은 없습니다."

# **3. 과제**

## 문제 1. 역할 부여 프롬프트로 진료 메모 요약하기

* `memo_1`을 아래 조건으로 요약하세요.
    * 역할: 환자에게 결과를 안내해야 하는 **간호사**
    * 조건: 전문 용어를 최소화하고, 환자가 이해하기 쉬운 표현으로 3문장 이내 요약
* 시스템 메시지에 역할과 조건을 함께 담아 프롬프트를 설계하세요.

**💡 풀이 포인트** — 시스템 메시지(`SystemMessage`)는 AI에게 "너는 누구이고, 어떤 규칙으로 답해야 하는지"를 알려주는 **업무 지침서**이고, 휴먼 메시지(`HumanMessage`)는 실제 **요청서**입니다. 역할(간호사)과 조건(쉬운 말·3문장 이내)을 지침서에 함께 적어 두면, 요청이 같아도 그 규칙을 지키며 답하게 됩니다. 첫 번째 셀은 프롬프트를 만드는 부분이라 API 키 없이 실행되고, 두 번째 셀(🔑)에서 실제로 GPT를 호출합니다.

In [3]:
# [문제 1] 프롬프트 설계 ① — 시스템 메시지에 '역할 + 조건'을 함께 적습니다
system_prompt_1 = """당신은 환자에게 진료 결과를 안내하는 친절한 간호사입니다.
아래 진료 메모를 환자분께 직접 설명하듯이 요약해 주세요.

[조건]
1. 의학 전문 용어와 단위는 최소화하고, 꼭 필요하면 쉬운 말로 풀어서 설명하세요.
   (예: '148/92mmHg' → '혈압이 목표보다 조금 높게 나왔어요')
2. 환자에게 말하듯 따뜻한 존댓말로 작성하세요.
3. 전체 요약은 반드시 3문장 이내로 작성하세요.
"""

# 프롬프트 설계 ② — 휴먼 메시지에는 '요청 + 요약할 데이터(memo_1)'를 담습니다
user_prompt_1 = f"다음 진료 메모를 환자분께 안내할 수 있도록 요약해 주세요.\n{memo_1}"

# 시스템 → 휴먼 순서로 리스트에 담습니다 (이 리스트가 GPT에게 그대로 전달됩니다)
messages_1 = [
    SystemMessage(content=system_prompt_1),   # 역할·조건 (업무 지침서)
    HumanMessage(content=user_prompt_1),      # 실제 요청 + 데이터 (요청서)
]

# GPT에게 어떤 내용이 전달되는지 눈으로 확인합니다
for m in messages_1:        # 메시지를 하나씩 꺼내서
    print(f"[{m.type}]")    # 메시지 종류 (system / human)
    print(m.content)        # 메시지 내용

[system]
당신은 환자에게 진료 결과를 안내하는 친절한 간호사입니다.
아래 진료 메모를 환자분께 직접 설명하듯이 요약해 주세요.

[조건]
1. 의학 전문 용어와 단위는 최소화하고, 꼭 필요하면 쉬운 말로 풀어서 설명하세요.
   (예: '148/92mmHg' → '혈압이 목표보다 조금 높게 나왔어요')
2. 환자에게 말하듯 따뜻한 존댓말로 작성하세요.
3. 전체 요약은 반드시 3문장 이내로 작성하세요.

[human]
다음 진료 메모를 환자분께 안내할 수 있도록 요약해 주세요.

[진료 메모 - 고혈압]
환자는 3년 전 고혈압 진단을 받았으며, 현재 하루 1회 혈압약을 복용 중이다.
최근 방문 시 측정한 혈압은 148/92mmHg로 목표 수치보다 다소 높게 나타났다.
환자는 최근 업무 스트레스가 많고, 야근이 잦아 규칙적인 식사와 운동이 어렵다고 호소했다.
염분 섭취를 줄이고 주 3회 이상 유산소 운동을 병행할 것을 권고하였으며, 2주 후 재방문하여
혈압 변화를 다시 확인하기로 하였다.



In [ ]:
# 🔑 [Colab에서 실행] 설계한 프롬프트로 GPT를 호출합니다 (API 키 필요 — 이 노트북에서는 미실행)
import re   # 문장 수를 세기 위한 정규표현식 모듈

response_1 = chat.invoke(messages_1)   # 메시지 리스트를 GPT에게 보내고 답변(AIMessage)을 받음
print(response_1.content)              # 답변 객체에서 본문 글자(.content)만 출력

# 조건(3문장 이내)을 지켰는지 간단히 확인: 마침표·물음표·느낌표 뒤의 공백을 기준으로 문장을 나눔
sentences = [s for s in re.split(r"(?<=[.!?])\s+", response_1.content.strip()) if s]
print("\n문장 수:", len(sentences), "→", "조건 충족 ✅" if len(sentences) <= 3 else "3문장 초과 ⚠️ (조건을 더 강하게 적어 보세요)")

## 문제 2. 조건/출력형식 지정 프롬프트로 증상 카테고리 분류하기

* `symptom_text`를 분석해서 아래 3개 항목을 **딕셔너리 형식**으로 출력하도록 프롬프트를 설계하세요.
    * `category` : 증상이 속하는 진료과 (예: 호흡기내과, 이비인후과, 내분비내과 등)
    * `risk_level` : 위험도 ("상", "중", "하" 중 하나)
    * `reason` : 판단 근거 1문장
* 출력은 다른 설명 없이 딕셔너리 형식 문자열만 나오도록 지시하고, `ast.literal_eval`로 파싱하세요.

**💡 풀이 포인트** — 사람이 읽을 답이 아니라 **프로그램이 다시 쓸 데이터**를 받으려면 출력 형식을 아주 구체적으로 정해줘야 합니다. 그래서 키 이름, 허용 값("상/중/하"), 형식 예시를 시스템 메시지에 적고 "딕셔너리 말고는 아무것도 쓰지 말라"고 못 박았습니다. GPT의 답은 겉모습만 딕셔너리일 뿐 실제로는 **문자열(str)**이라서, `ast.literal_eval`(문자열을 안전하게 파이썬 자료형으로 바꾸는 함수, `eval`과 달리 코드를 실행하지 않아 안전함)로 진짜 딕셔너리(dict)로 바꿔야 `result['category']`처럼 꺼내 쓸 수 있습니다.

In [4]:
# [문제 2] 프롬프트 설계 — 출력 형식(딕셔너리)과 허용 값을 구체적으로 정해 줍니다
system_prompt_2 = """당신은 병원 접수처에서 환자의 증상을 보고 진료과와 위험도를 분류하는 의료 분류 보조 AI입니다.
환자의 증상 설명을 분석해서 아래 3개 키를 가진 파이썬 딕셔너리 형식으로만 답하세요.

- 'category': 증상이 속하는 진료과 1개 (예: 호흡기내과, 이비인후과, 내분비내과 등)
- 'risk_level': 위험도. 반드시 "상", "중", "하" 중 하나
- 'reason': 판단 근거 1문장

[출력 규칙]
- 딕셔너리 외의 설명, 인사말, 마크다운 코드블록(```)은 절대 쓰지 마세요.
- 아래 형식을 그대로 따르세요.
{'category': '진료과', 'risk_level': '상/중/하 중 하나', 'reason': '판단 근거 1문장'}
"""

messages_2 = [                                                # GPT에게 보낼 메시지 리스트
    SystemMessage(content=system_prompt_2),                   # 형식·규칙 지침
    HumanMessage(content=f"[증상 설명]\n{symptom_text}"),     # 분석할 증상 텍스트
]

def parse_dict_output(text):
    """GPT 답변 문자열에서 { } 부분만 잘라 ast.literal_eval로 진짜 딕셔너리로 바꿉니다."""
    start = text.find("{")                         # 첫 번째 여는 중괄호 위치
    end = text.rfind("}")                          # 마지막 닫는 중괄호 위치
    if start == -1 or end == -1:                   # 중괄호가 없으면 형식이 잘못된 답변
        raise ValueError("딕셔너리 형식을 찾지 못했습니다:\n" + text)   # 오류를 내서 문제를 바로 알 수 있게 함
    return ast.literal_eval(text[start:end + 1])   # 앞뒤에 ``` 같은 군더더기가 붙어도 중괄호 안쪽만 변환

# (참고) 파서가 잘 동작하는지 '직접 만든 테스트 문자열'로 확인합니다 — GPT 답변이 아닙니다
test_text = "{'category': '예시과', 'risk_level': '하', 'reason': '파서 동작 확인용 문장입니다.'}"
test_result = parse_dict_output(test_text)        # 문자열 → 딕셔너리
print(type(test_text), "→", type(test_result))    # str → dict 로 바뀌었는지 확인
print("risk_level 값 꺼내기:", test_result["risk_level"])   # 이제 키로 값을 꺼낼 수 있습니다

<class 'str'> → <class 'dict'>
risk_level 값 꺼내기: 하


In [ ]:
# 🔑 [Colab에서 실행] GPT 호출 → 문자열 답변을 딕셔너리로 파싱 (API 키 필요)
response_2 = chat.invoke(messages_2)               # GPT 호출
print("GPT 원본 답변:", response_2.content)         # 파싱 전에는 그냥 문자열
print("원본 타입:", type(response_2.content))       # <class 'str'>

result_2 = parse_dict_output(response_2.content)   # ast.literal_eval로 문자열 → 딕셔너리
print("\n파싱 결과 타입:", type(result_2))           # <class 'dict'>
for key, value in result_2.items():                # 키-값을 하나씩 꺼내서 출력
    print(f"{key}: {value}")

# 위험도 값이 규칙("상"/"중"/"하")을 지켰는지 확인
print("\nrisk_level 형식:", "OK ✅" if result_2["risk_level"] in ["상", "중", "하"] else "규칙 위반 ⚠️")

## 문제 3. 마크다운 문법 프롬프트로 복약 안내문 생성하기

* `memo_1`(고혈압) 환자를 위한 복약 안내문을 아래 마크다운 구조로 생성하도록 프롬프트를 설계하세요.
    * `#` : 안내문 제목
    * `##` : 소제목 (복약 방법 / 생활습관 안내 / 주의사항)
    * `-` : 각 소제목별 bullet point 설명
    * `**...**` : 꼭 지켜야 할 핵심 내용 강조

**💡 풀이 포인트** — 마크다운(Markdown)은 `#`, `-`, `**` 같은 간단한 기호로 제목·목록·강조를 표시하는 문서 작성 문법입니다. 원하는 구조(제목 1개, 소제목 3개의 이름과 순서, bullet, 강조)를 프롬프트에 **빈칸이 있는 뼈대 그대로** 보여주면, GPT가 그 틀을 채우는 방식으로 답해서 결과가 안정적입니다. 진료 메모에 약 이름이 없으므로 "없는 약 이름·용량은 지어내지 말라"는 규칙도 넣었습니다(의료 안내문에서는 꼭 필요한 안전장치예요).

In [5]:
# [문제 3] 프롬프트 설계 — 원하는 마크다운 '뼈대'를 그대로 보여 줍니다
system_prompt_3 = """당신은 고혈압 환자에게 복약 안내문을 만들어 주는 병원 약사입니다.
진료 메모를 참고해서 환자용 복약 안내문을 아래 마크다운 구조 그대로 작성하세요.

# (안내문 제목)
## 복약 방법
- (설명)
## 생활습관 안내
- (설명)
## 주의사항
- (설명)

[작성 규칙]
- '#'은 제목 1개에만, '##'은 위의 소제목 3개(복약 방법 / 생활습관 안내 / 주의사항)에만 사용하세요.
- 각 소제목 아래 설명은 모두 '-'로 시작하는 bullet point로 2~4개씩 작성하세요.
- 꼭 지켜야 할 핵심 내용은 **굵게** 강조하세요.
- 진료 메모에 없는 약 이름이나 용량은 지어내지 말고, '처방받은 대로' 복용하도록 안내하세요.
- 환자가 이해하기 쉬운 말과 존댓말로 작성하세요.
"""

messages_3 = [
    SystemMessage(content=system_prompt_3),   # 역할 + 마크다운 구조 + 규칙
    HumanMessage(content=f"[진료 메모]\n{memo_1}\n위 환자를 위한 복약 안내문을 작성해 주세요."),   # 요청 + 데이터
]

print(messages_3[0].content)   # 시스템 메시지에 담긴 구조와 규칙 확인

당신은 고혈압 환자에게 복약 안내문을 만들어 주는 병원 약사입니다.
진료 메모를 참고해서 환자용 복약 안내문을 아래 마크다운 구조 그대로 작성하세요.

# (안내문 제목)
## 복약 방법
- (설명)
## 생활습관 안내
- (설명)
## 주의사항
- (설명)

[작성 규칙]
- '#'은 제목 1개에만, '##'은 위의 소제목 3개(복약 방법 / 생활습관 안내 / 주의사항)에만 사용하세요.
- 각 소제목 아래 설명은 모두 '-'로 시작하는 bullet point로 2~4개씩 작성하세요.
- 꼭 지켜야 할 핵심 내용은 **굵게** 강조하세요.
- 진료 메모에 없는 약 이름이나 용량은 지어내지 말고, '처방받은 대로' 복용하도록 안내하세요.
- 환자가 이해하기 쉬운 말과 존댓말로 작성하세요.



In [ ]:
# 🔑 [Colab에서 실행] GPT 호출 후 마크다운으로 렌더링해서 확인 (API 키 필요)
from IPython.display import Markdown, display   # 마크다운 문자열을 서식 있는 문서로 보여주는 도구

response_3 = chat.invoke(messages_3)   # GPT 호출
print(response_3.content)              # ① 원본: #, -, ** 기호가 그대로 보이는 텍스트
display(Markdown(response_3.content))  # ② 렌더링: 제목·목록·굵은 글씨가 적용된 안내문

## 문제 4. 같은 입력에 역할만 바꿔서 결과 비교하기

* `memo_2`(당뇨) 환자 메모에 대해 같은 질문을 하되, 시스템 메시지의 **역할만** 아래 2가지로 바꿔서 결과를 비교하세요.
    * 역할 A: 환자와 오래 알고 지낸 **친근한 담당 의사**
    * 역할 B: 수치와 근거 중심으로 설명하는 **전문 내분비내과 전문의**
* 질문은 두 경우 모두 동일: "이 환자에게 지금 가장 신경 써야 할 점을 알려줘"

**💡 풀이 포인트** — 비교 실험에서는 **딱 하나만** 바꿔야 차이의 원인을 알 수 있습니다(실험의 '통제 변인'과 같은 원리예요). 그래서 휴먼 메시지(진료 메모 + 질문)는 변수 하나로 만들어 두 경우에 똑같이 쓰고, 시스템 메시지에는 다른 조건 없이 **역할 문장만** 넣었습니다. 두 번째 셀에서는 답변을 나란히 출력하고, 검사 수치(HbA1c 7.4%, 공복혈당 138mg/dL)를 직접 언급했는지 같은 간단한 지표도 함께 찍어서 '말투 외의 내용 차이'를 확인할 수 있게 했습니다.

In [6]:
# [문제 4] 두 경우 모두 똑같이 쓸 질문과 휴먼 메시지 (고정)
question_4 = "이 환자에게 지금 가장 신경 써야 할 점을 알려줘"
human_msg_4 = HumanMessage(content=f"[진료 메모]\n{memo_2}\n\n[질문]\n{question_4}")

# 역할만 다른 시스템 메시지 2개 (다른 조건은 일부러 넣지 않음)
system_A = SystemMessage(content="당신은 이 환자와 오래 알고 지낸 친근한 담당 의사입니다.")             # 역할 A
system_B = SystemMessage(content="당신은 수치와 근거를 중심으로 설명하는 전문 내분비내과 전문의입니다.")   # 역할 B

messages_A = [system_A, human_msg_4]   # 역할 A + 같은 질문
messages_B = [system_B, human_msg_4]   # 역할 B + 같은 질문

# 정말 '역할만' 다른지 확인
print("역할 A:", messages_A[0].content)
print("역할 B:", messages_B[0].content)
print("두 경우 휴먼 메시지가 같은 객체인가?", messages_A[1] is messages_B[1])   # 같은 변수를 썼으므로 True

역할 A: 당신은 이 환자와 오래 알고 지낸 친근한 담당 의사입니다.
역할 B: 당신은 수치와 근거를 중심으로 설명하는 전문 내분비내과 전문의입니다.
두 경우 휴먼 메시지가 같은 객체인가? True


In [ ]:
# 🔑 [Colab에서 실행] 같은 질문을 두 역할로 각각 호출해서 비교 (API 키 필요)
answer_A = chat.invoke(messages_A).content   # 역할 A의 답변 본문
answer_B = chat.invoke(messages_B).content   # 역할 B의 답변 본문

print("=" * 25, "역할 A: 친근한 담당 의사", "=" * 25)
print(answer_A)
print("\n" + "=" * 25, "역할 B: 내분비내과 전문의", "=" * 25)
print(answer_B)

# 말투 말고 '내용' 차이를 보기 위한 간단한 지표
print("\n[비교 지표]")
for name, ans in [("A", answer_A), ("B", answer_B)]:            # 두 답변을 차례로
    mentions_numbers = any(x in ans for x in ["7.4", "138"])    # 검사 수치를 직접 언급했는지
    print(f"역할 {name} | 글자 수: {len(ans)} | 검사 수치 언급: {'예' if mentions_numbers else '아니오'}")

**질문**: 두 결과에서 말투 외에 실제로 다루는 내용(강조하는 포인트)에도 차이가 있나요? 어떤 상황에서 각 역할 설정이 더 적합할지 한 문장으로 정리해보세요.

**답변 (조태희)**

> ✍️ 아래는 두 시스템 메시지의 차이로부터 **예상되는 경향**을 먼저 정리한 것입니다. Colab에서 위 셀을 실행한 뒤 실제 두 답변과 [비교 지표]를 보고, 예상과 같은 점·다른 점을 한두 문장 덧붙여 주세요.

네, 말투뿐 아니라 강조하는 내용도 달라질 가능성이 큽니다. 역할 A(친근한 담당 의사)는 "야식과 단 음료 줄이기", "혈당을 더 자주 재 보기"처럼 **생활 속에서 바로 실천할 행동과 격려**에 초점을 맞추기 쉽고, 역할 B(내분비내과 전문의)는 HbA1c 7.4%·공복혈당 138mg/dL 같은 **검사 수치가 무엇을 의미하는지, 자가측정 빈도와 치료 조정이 왜 필요한지** 같은 근거 중심 내용을 앞세우기 쉽습니다. 시스템 메시지의 역할 한 줄이 "무엇을 중요하게 볼지"까지 바꾸기 때문입니다.

**한 문장 정리:** 환자가 부담 없이 생활습관을 바꾸도록 동기를 줘야 하는 일상 상담에는 역할 A가, 수치 변화를 정확히 해석해 치료 계획을 세우거나 의료진끼리 정보를 전달해야 하는 상황에는 역할 B가 더 적합합니다.